# Data Lineage & Compliance

Every AI-generated summary used in a legal proceeding must be **reproducible and traceable**. If a defense attorney asks "where did this summary come from?", the system must answer with:
- The exact source document version (byte-for-byte)
- The model configuration used
- The confidence score at generation time
- Whether it passed all guardrails
- Whether the evidence reference was validated before summarization

This notebook demonstrates immutable data lineage using the Briefcase AI SDK:

| Capability | SDK Module | What It Proves |
|-----------|-----------|---------------|
| Reference validation | `PromptValidationEngine` | Evidence references are verified before LLM calls |
| Version-linked decisions | `DataRef` + content hashing | Every output traces to its exact input |
| Drift detection | `DriftCalculator` + `emit_drift_detected` | Quantifies and reports output changes |
| Compliance reporting | `SOC2ReportGenerator` | Automated control evaluation over decision history |

> All demos run fully offline. In production, `lakeFS` replaces content hashing with commit SHAs.

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import hashlib
from pathlib import Path
from datetime import datetime, timedelta

from briefcase.drift import DriftCalculator
from briefcase.compliance.reports.soc2 import SOC2ReportGenerator
from briefcase.events.emitter import emit_drift_detected
from src.mock_llm import MockLLMProvider
from src.pipeline import summarize_report
from src.config import storage, event_bus
from src.validation import validate_prompt

print("Lineage, compliance, validation, and event modules loaded")

---
## Part 1: Version-Linked Decisions

Every `DecisionSnapshot` includes a `data_version` tag — a SHA-256 content hash of the source document. Before summarization, the `PromptValidationEngine` validates that evidence references actually exist.

In production with lakeFS, the content hash becomes a commit SHA with full branch history. The pattern is identical.

In [ ]:
# Validate evidence references before summarization
v1_check = validate_prompt("Summarize report_001")
v2_check = validate_prompt("Summarize report_001_amended")

print(f"Reference validation:")
print(f"  report_001:         {v1_check.status} ({v1_check.references_checked} refs checked)")
print(f"  report_001_amended: {v2_check.status} ({v2_check.references_checked} refs checked)")

# Load original and amended versions of report_001
original_text = Path("data/police_reports/report_001.txt").read_text()
amended_text = Path("data/police_reports/report_001_amended.txt").read_text()

original_hash = hashlib.sha256(original_text.encode()).hexdigest()[:12]
amended_hash = hashlib.sha256(amended_text.encode()).hexdigest()[:12]

print(f"\nreport_001.txt:")
print(f"  Length:  {len(original_text)} chars")
print(f"  Hash:    {original_hash}")
print(f"\nreport_001_amended.txt:")
print(f"  Length:  {len(amended_text)} chars")
print(f"  Hash:    {amended_hash}")
print(f"\nSame document? {original_hash == amended_hash}")
print(f"Difference:    {len(amended_text) - len(original_text)} chars added")

### What Changed?

The amended report adds a detective's addendum with critical new information: an updated suspect description and an AFIS fingerprint match. This is exactly the kind of evidence update that can change the direction of a case.

In [ ]:
# Show the addendum that was appended
addendum_start = amended_text.find("ADDENDUM")
if addendum_start > 0:
    print("Added section (detective addendum):\n")
    print(amended_text[addendum_start:])

### Summarize Both Versions

The pipeline produces different summaries for different input versions — and each summary is linked to the exact document hash that produced it.

In [ ]:
llm = MockLLMProvider(model="gpt-4o", simulate_latency=False)

# Summarize original
result_v1 = await summarize_report("report_001", original_text, llm=llm)
# Summarize amended
result_v2 = await summarize_report("report_001_amended", amended_text, llm=llm)

print(f"{'Metric':<20} {'v1 (original)':>20} {'v2 (amended)':>20}")
print("─" * 62)
print(f"{'Data version':<20} {result_v1['data_version']:>20} {result_v2['data_version']:>20}")
print(f"{'Snapshot ID':<20} {result_v1['snapshot_id'][:16] + '...':>20} {result_v2['snapshot_id'][:16] + '...':>20}")
print(f"{'Confidence':<20} {result_v1['confidence']:>20.2f} {result_v2['confidence']:>20.2f}")
print(f"{'Summary length':<20} {len(result_v1['summary']):>20} {len(result_v2['summary']):>20}")
print(f"\nSame summary? {result_v1['summary'] == result_v2['summary']}")

### Comparing the Summaries

The v2 summary should include the new information from the addendum — the updated suspect description and the AFIS match to Marcus Webb.

In [ ]:
print("=" * 70)
print("v1 SUMMARY (original report):")
print("=" * 70)
print(result_v1["summary"])

print(f"\n{'=' * 70}")
print("v2 SUMMARY (with detective addendum):")
print("=" * 70)
print(result_v2["summary"])

# Highlight what's new in v2
v2_lower = result_v2["summary"].lower()
new_details = []
for term in ["carlos rivera", "marcus webb", "6 feet", "tattoo", "honda civic", "warrant"]:
    if term in v2_lower and term not in result_v1["summary"].lower():
        new_details.append(term)

if new_details:
    print(f"\nNew details in v2: {', '.join(new_details)}")

---
## Part 2: Drift Detection

When source data changes, how much does the output change? `DriftCalculator` quantifies this — essential for monitoring a pipeline where evidence documents get updated, corrected, or supplemented over time.

In [ ]:
drift_calc = DriftCalculator()

# Compare v1 and v2 summaries
drift = drift_calc.calculate_drift([result_v1["summary"], result_v2["summary"]])

print(f"Drift Analysis: report_001 original vs. amended")
print(f"{'─' * 50}")
print(f"  Drift score:       {drift.drift_score:.3f}  (0=identical, 1=completely different)")
print(f"  Consistency score: {drift.consistency_score:.3f}")
print(f"  Agreement rate:    {drift.agreement_rate:.3f}")

if drift.drift_score > 0.3:
    print(f"\n  Significant drift detected — the addendum materially changed the summary")

# Emit structured drift event
await emit_drift_detected(None, {
    "drift_score": drift.drift_score,
    "consistency_score": drift.consistency_score,
    "source": "lineage_comparison",
})
print(f"\n  [event] drift.detected emitted")

### Cross-Model Drift

How much do GPT-4o and Claude Sonnet differ on the **same input**? This is model behavioral drift — important for deciding whether models are interchangeable.

In [ ]:
# Get summaries from both models for each report
report_ids = ["report_001", "report_002", "report_003", "report_004", "report_005"]

gpt4o = MockLLMProvider(model="gpt-4o", simulate_latency=False)
claude = MockLLMProvider(model="claude-sonnet", provider="anthropic", simulate_latency=False)

print(f"{'Report':<12} {'Drift Score':>12} {'Consistency':>12}")
print("─" * 40)

for rid in report_ids:
    summary_gpt = (await gpt4o.generate(rid, "Summarize"))["summary"]
    summary_claude = (await claude.generate(rid, "Summarize"))["summary"]
    
    d = drift_calc.calculate_drift([summary_gpt, summary_claude])
    print(f"{rid:<12} {d.drift_score:>12.3f} {d.consistency_score:>12.3f}")

---
## Part 3: Lineage Audit Trail

Every stored decision carries enough metadata to reconstruct exactly what happened. Let's query the storage backend and verify the audit trail.

In [ ]:
# Load both decisions and verify lineage fields
for label, sid in [("v1 (original)", result_v1["snapshot_id"]), ("v2 (amended)", result_v2["snapshot_id"])]:
    loaded = storage.load_decision(sid)
    tags = loaded.tags
    
    print(f"\n{label}:")
    print(f"  Snapshot ID:   {sid}")
    print(f"  Function:      {loaded.function_name}")
    print(f"  Data version:  {tags.get('data_version', 'N/A')}")
    print(f"  Report ID:     {tags.get('report_id', 'N/A')}")
    print(f"  Data ref URI:  {tags.get('data_ref_uri', 'N/A')}")
    print(f"  Exec time:     {loaded.execution_time_ms:.2f}ms")
    print(f"  Output conf:   {loaded.outputs[0].confidence}")

print(f"\n{'─' * 50}")
print(f"Both decisions are independently traceable to their")
print(f"exact source document version via the data_version tag.")

---
## Part 4: SOC2 Compliance Reporting

The `SOC2ReportGenerator` evaluates compliance controls over the decision history. This is required for CIS-compliant deployments in county IT environments.

The report evaluates controls like:
- **CC6.1**: Logical and Physical Access Controls
- **CC7.2**: System Monitoring
- And others from the SOC2 Type II framework

In [ ]:
generator = SOC2ReportGenerator(None)

report = generator.evaluate(
    "evidence-poc",          # engagement ID
    "main",                  # workstream
    datetime.now() - timedelta(days=7),
    datetime.now(),
)

print(f"SOC2 Compliance Report")
print(f"{'─' * 50}")
print(f"  Organization:    {report.organization}")
print(f"  Framework:       {report.framework}")
print(f"  Report period:   {report.report_period_start:%Y-%m-%d} to {report.report_period_end:%Y-%m-%d}")
print(f"  Overall status:  {report.overall_status}")
print(f"  Overall score:   {report.overall_score}%")
print(f"  Controls:        {report.total_controls_evaluated} evaluated")
print(f"    Passed:        {report.controls_passed}")
print(f"    Failed:        {report.controls_failed}")
print(f"    Partial:       {report.controls_partial}")
print(f"  Violations:      {len(report.violations)}")

In [ ]:
# Show the full control results
print("Control Results:\n")
for control in report.control_results:
    status_icon = {
        "COMPLIANT": "[PASS]",
        "NON_COMPLIANT": "[FAIL]",
        "PARTIALLY_COMPLIANT": "[WARN]",
    }.get(control.status, "[????]")
    
    print(f"  {status_icon} {control.control_id}: {control.control_name}")
    print(f"         Score: {control.score}% | Status: {control.status}")
    if hasattr(control, 'findings') and control.findings:
        for f in control.findings[:2]:
            print(f"         Finding: {f}")
    print()

In [ ]:
# Generate the full markdown report (what you'd share with compliance)
md_report = report.to_markdown()

print("Full SOC2 Markdown Report (first 1000 chars):\n")
print(md_report[:1000])
print("\n... [truncated]")
print(f"\nFull report length: {len(md_report)} characters")

---
## Part 5: Production Architecture (lakeFS Integration)

In production, content hashing is replaced by lakeFS version control. The SDK's `PromptValidationEngine` accepts a `lakefs_client` that resolves references against versioned data. Here we demonstrate the pattern with a mock client — the same code works with a real lakeFS server by swapping the client.

In [ ]:
from src.validation import MockLakeFSClient, create_validation_engine

# In production, this would be:
# from briefcase.integrations.lakefs import VersionedClient
# client = VersionedClient(endpoint="lakefs.internal:8000", access_key=os.environ["LAKEFS_KEY"])

# For the POC, we use a mock that returns a deterministic commit SHA
mock_client = MockLakeFSClient()
commit_sha = mock_client.get_commit("evidence-repo", "main")
print(f"lakeFS commit SHA: {commit_sha}")

# The validation engine uses this client to stamp every validation report
engine = create_validation_engine()
report = engine.validate("Summarize report_001")
print(f"\nValidation report:")
print(f"  Status:       {report.status}")
print(f"  Refs checked: {report.references_checked}")
print(f"  lakeFS commit:{report.lakefs_commit}")
print(f"  Time:         {report.validation_time_ms:.2f}ms")

# In production, swapping MockLakeFSClient for VersionedClient is the only change
# The pipeline, guardrails, replay, routing, and lineage code stays identical
print(f"\nPattern: MockLakeFSClient -> VersionedClient (zero code changes in pipeline)")

## Key Takeaways

- **Reference validation**: `PromptValidationEngine` verifies evidence references exist before the LLM is called — catches invalid refs early
- **Immutable lineage**: Every `DecisionSnapshot` links to its exact source document version via content hash (or lakeFS commit SHA in production)
- **Drift is quantified and reported**: `DriftCalculator` measures output change (0.0-1.0 scale), `emit_drift_detected` notifies downstream systems
- **Compliance is automated**: `SOC2ReportGenerator` produces audit-ready reports with control-by-control evaluation
- **Audit trail is complete**: Given a snapshot ID, you can reconstruct the exact input, model config, output, confidence, and data version
- **Production-ready pattern**: Content hashing + `MockLakeFSClient` in the POC maps directly to lakeFS commit SHAs + `VersionedClient` — same code, different backend

**This is the full capability set**: Notebooks 01-04 demonstrate instrumentation, evaluation, error reproduction, and lineage — the four gaps Alex identified in his current workflow.